# Data Preparation

It is important to run this code block before working on the rest of the notebook. Check out the comments to help you understand what each step does.

Refer to the [VCQ code book](https://docs.google.com/document/d/1uYfUuUsPVQyD3puVvbIMjamYuVPr_f4gtJuSB5Wj-WA/edit) for questions that match the columns in the following dataset.

In [1]:
# Step 1: Collect the original data into a DataFrame object
import pandas as pd
raw_df = pd.read_csv('https://raw.githubusercontent.com/qarnac/cs201/main/vcq_data_nan.csv')

# Step 2: Create a DataFrame object to hold only columns from the family satisfaction
# and deviance sections
dev_fam_df = raw_df.filter(regex='^deviance|^family').copy()

# Step 3: Recode responses for negatively worded questions
r_columns = ['deviance_play', 'deviance_cautious', 'family_stress', 'family_secrets', 'family_fight']
dev_fam_df[r_columns] = 7 - dev_fam_df[r_columns]

# Step 4: Determine and save section means
family_columns = dev_fam_df.filter(regex='^family')
deviance_columns = dev_fam_df.filter(regex='^deviance')
dev_fam_df['family_mean'] = family_columns.mean(axis = 1)
dev_fam_df['deviance_mean'] = deviance_columns.mean(axis = 1)

# Step 5: Create a new DataFrame to combine section means with demographic data
data = {'gender':raw_df['demo_gender'].copy(),
        'relationship': raw_df['demo_relationshipstatus01'].copy(),
        'children': raw_df['demo_children'],
        'classstanding': raw_df['demo_classstanding'],
        'past_employed': raw_df['demo_employed01'],
        'current_employed': raw_df['demo_employed02'],
        'politics': raw_df['demo_politics'],
        'religion': raw_df['demo_religion'],
        'family_satisfaction': dev_fam_df['family_mean'],
        'deviance': dev_fam_df['deviance_mean']}
df = pd.DataFrame(data)

# Step 6: replace numbers with text for categorical columns
gender_map = {1:'Male', 2:'Female', 3:'Other'}
df['gender'] = df['gender'].replace(gender_map)
df['relationship'] = df['relationship'].replace({1:'No', 2:'Yes'})
yes_no_columns = ['children', 'past_employed', 'current_employed']
yes_no_map = {1:'Yes', 2:'No'}
df[yes_no_columns] = df[yes_no_columns].replace(yes_no_map)
class_labels = ['First-year', 'Sophomore', 'Junior', 'Senior']
class_bins = [0, 1, 2, 3, 4]
df['classstanding'] = pd.cut(df['classstanding'],
                             bins=class_bins,
                             labels=class_labels,
                             right=True)
politics_labels = ['conservative', 'moderate', 'liberal']
politics_bins = [0, 2, 4, 6]
df['politics'] = pd.cut(df['politics'],
                             bins=politics_bins,
                             labels=politics_labels,
                             right=True)

religion_labels = ['Yes', 'No']
religion_bins = [0, 3, 6]
df['religion'] = pd.cut(df['religion'],
                             bins=religion_bins,
                             labels=religion_labels,
                             right=True)


df.head(10)


,gender,relationship,children,classstanding,past_employed,current_employed,politics,religion,family_satisfaction,deviance
0,Male,No,No,Sophomore,Yes,Yes,NaN,No,4.285714,3.625
1,Female,Yes,No,Junior,Yes,Yes,moderate,No,4.857143,3.125
2,Male,No,No,Junior,No,No,moderate,Yes,5.142857,3.125
3,Female,No,No,Sophomore,No,No,moderate,No,4.714286,3.500
4,Female,No,No,Sophomore,No,No,moderate,Yes,4.428571,3.625
5,Female,No,No,Junior,Yes,Yes,NaN,Yes,3.714286,3.125
6,Male,Yes,No,Junior,Yes,Yes,conservative,Yes,4.714286,2.750
7,Female,No,No,Senior,Yes,Yes,liberal,Yes,4.714286,3.750
8,Female,Yes,No,Junior,Yes,Yes,moderate,Yes,5.000000,3.375
9,Male,Yes,No,Senior,Yes,Yes,liberal,No,4.000000,3.125


# The groupby function

The `groupby()` function in pandas is used to split your data into groups based on the values in one or more columns, and then apply a calculation (like mean, count, sum) to each group. Think of it like sorting your data into labeled bins, and then doing math inside each bin.

For example, the following call to the `groupby` function groups values in the `'family_satisfaction'` column based on `'gender'` and then **count** the number of respondents within each gender.

In [ ]:
df.groupby('gender')['family_satisfaction'].count()

,family_satisfaction
gender,
Female,362
Male,176
Other,4


We can see that out of the 542 students who answered the family satisfaction section of the survey, 362 of them were female, 176 were male, and 4 answered other.

The following call to the groupby function groups values in the 'family_satisfaction' column based on 'gender' and the calculate the average family satisfaction within each gender.

In [ ]:
df.groupby('gender')['family_satisfaction'].mean()

,family_satisfaction
gender,
Female,4.360695
Male,4.263528
Other,4.214286


We can see that the average family satisfaction for female was a bit higher than the average family satisfaction for male. The average family satisfaction for other was the lowest.

Using the same approach, we can also find out the standard deviation of family satisfaction within each gender.

In [ ]:
df.groupby('gender')['family_satisfaction'].std()

,family_satisfaction
gender,
Female,0.763544
Male,0.815081
Other,0.965387


We can see that female has the least variation within their group while other has the highest variation within their group.

# The agg function

The `agg()` function stands for *aggregate*, and it lets you apply multiple statistics functions (like mean, count, std, min, max, median, etc.) to your grouped data.

In [ ]:
df.groupby('gender', observed=True)['family_satisfaction'].agg(['count', 'mean', 'std', 'min', 'max', 'median'])

,count,mean,std,min,max,median
gender,,,,,,
Female,362,4.360695,0.763544,1.000000,6.000000,4.428571
Male,176,4.263528,0.815081,2.000000,5.857143,4.285714
Other,4,4.214286,0.965387,3.142857,5.428571,4.142857


From the above results, we can see several statistics in one table format. For example,
*  362 female students and 176 male students answered the family satisfaction section.
*  Only 4 students marked other for gender on their survey. With such a small number, it would be hard to generalize any findings for this group.
*  The lowest level of family satisfaction for female students was 1 while the the highest level was 6.
*  The median and average family satisfaction for female students were slightly higher than those for male students. Meanwhile, responses from female students varied slightly less than responses from male students.


## Practice

In [ ]:
# Group 'family_satisfaction' using a different demographic group and use the agg function
# to show the count, mean, and std within each group


In [ ]:
# Group 'deviance' using a demographic group of your choice and use the agg function
# to show the count, mean, and std within each group


**Add a text block** to communicate what you learned from the above results.

# Pivot Tables

A pivot table is a powerful tool used to summarize, organize, and analyze data, especially when working with large datasets. It allows you to group data by multiple categories and compute statistics for each sub-group.

A `pivot_table` function call typically specifies three parameters:

*  **values**: This identifies the numeric column(s) on which we want to perform analyses.
*  **index**: This specifies the groupings that will be performed.
*  **aggfunc**: This specifies the analysis to be performed.

In the following code block, we pivot the `family_satisfaction` column using the `gender` column to perform the count action.

In [ ]:
pivot_by_gender = df.pivot_table(values='family_satisfaction',
              index='gender',
              aggfunc = 'count')
pivot_by_gender

,family_satisfaction
gender,
Female,362
Male,176
Other,4


The above pivot table generates similar results as the use of groupby function.

For each of the three parameters, we are allowed to specify a list.

The following block shows how we can specify two numeric columns to be analyzed: 'family_satisfaction' and 'deviance'.

In [ ]:
pivot_by_gender_class = df.pivot_table(values=['family_satisfaction', 'deviance'],
                        index='gender',
                        aggfunc = 'mean',
                        observed=True)
pivot_by_gender_class

,deviance,family_satisfaction
gender,,
Female,3.143897,4.360695
Male,3.438920,4.263528
Other,3.500000,4.214286


From the above pivot table, we can see that
*  average deviance for female is lower than male
*  average family satisfaction for female is higher than male

The following example shows how we can ask the analyses be grouped by two categorical columns: first by 'gender' and then by 'classstanding' within each gender.

In [ ]:
pivot_by_gender_class = df.pivot_table(values=['family_satisfaction', 'deviance'],
                        index=['gender', 'classstanding'],
                        aggfunc = 'mean',
                        observed=True)
pivot_by_gender_class

deviance  family_satisfaction
gender classstanding                               
Female First-year     2.978741             4.551020
       Sophomore      3.161585             4.246806
       Junior         3.111919             4.366667
       Senior         3.265542             4.530997
Male   First-year     3.375000             4.351648
       Sophomore      3.260204             4.212828
       Junior         3.546905             4.254603
       Senior         3.477106             4.315018
Other  First-year     4.375000             3.857143
       Sophomore      4.125000             3.142857
       Junior         2.750000             4.928571

Here are a few things we can observe from the above pivot table:

*  For female students, first year respondents had lowest average deviance and highest average family satisfaction.
*  For male students, sophomore respondents had lowest average deviance and lowest average family satisfaction.
*  Female students had higher average family satisfaction than male in every class standing.

The following code block shows that we can also perform multiple analyses: mean and standard deviation.

In [ ]:
pivot_by_gender_class = df.pivot_table(values=['family_satisfaction', 'deviance'],
                        index=['gender', 'classstanding'],
                        aggfunc = ['mean', 'std'],
                        observed=True)
pivot_by_gender_class

mean                           std  \
                      deviance family_satisfaction  deviance   
gender classstanding                                           
Female First-year     2.978741            4.551020  0.602176   
       Sophomore      3.161585            4.246806  0.702150   
       Junior         3.111919            4.366667  0.662910   
       Senior         3.265542            4.530997  0.655159   
Male   First-year     3.375000            4.351648  0.762056   
       Sophomore      3.260204            4.212828  0.710247   
       Junior         3.546905            4.254603  0.772003   
       Senior         3.477106            4.315018  0.596334   
Other  First-year     4.375000            3.857143       NaN   
       Sophomore      4.125000            3.142857       NaN   
       Junior         2.750000            4.928571  0.176777   

                                          
                     family_satisfaction  
gender classstanding                      
Female First-year               0.701726  
       Sophomore                0.803557  
       Junior                   0.718824  
       Senior                   0.799027  
Male   First-year               0.596519  
       Sophomore                0.875849  
       Junior                   0.806509  
       Senior                   0.838250  
Other  First-year                    NaN  
       Sophomore                     NaN  
       Junior                   0.707107

As we can see from the above result, performing multiple analyses on multiple columns using multiple grouping can make the resulting pivot table complex and hard to read.

## Practice

In [ ]:
# Create a pivot table on the count, average, and standard deviation
# for family satisfaction based on a categorical column that is neigher gender
# nor classstanding.



In [ ]:
# Create a pivot table on average deviance using two demographics columns.
# At least one of the columns must be different from gender or classstanding.



In [ ]:
# Create a pivot table of your choice that's different from what have been done.



**Add a text block** to communicate your findings about the students based on the above results.